# Lab 14: Project Readiness

**Business question:** Which recommendation follows from the evidence, and which uncertainty remains?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | A worked example on the course complaints | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | Your own project (the course data if you have none yet) | about 30 min | Three exercises and the interpretation |
| 3 Challenge | Either | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

Keep the **Lab Companion** for Module 14 open beside this notebook. It explains each step and shows the output you should expect.

This week introduces no new method. You practise the checks that make a project result trustworthy. Rerun it from a fresh state, and record files, versions and seeds. Show each number with its **denominator** (the number of cases it is counted out of) and its **uncertainty**. Part 1 shows the workflow on a small worked example. Part 2 applies it to your own project.

## Part 1 Guided core

### 1.1 Import the libraries and note their versions

Select the `ban6403` kernel in VS Code, then run the cell below. A result that someone else can rerun names the package versions it used. The dictionary `versions` keeps them for the record you save in step 1.15.

In [ ]:
# Path builds file paths; hashlib computes fingerprints; platform reports the Python version;
# textwrap breaks long titles into lines.
from pathlib import Path
import hashlib
import json
import platform
import re
import textwrap
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# The Python and package versions, kept for the manifest in step 1.15.
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}
versions

### 1.2 Set the output folder

The notebook runs from the `notebooks` folder, so the course data is one folder up, in `data`. Your saved files go to `outputs/m14`.

In [ ]:
# ".." means one folder up from notebooks.
OUTPUT_DIR = Path("../outputs/m14")
# parents=True also creates the outputs folder if needed; exist_ok=True means no error if the folder exists.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 The worked example: data, seed and data fingerprint

**The case.** The operations lead for credit cards at a bank asks: *should complaints that mention a fee get a priority review?* One piece of evidence is how such complaints were closed. Did the company give money back (`Closed with monetary relief`) more often when the complaint mentions a fee?

Before any analysis, record two things:

- the **data file** and its **fingerprint**. A SHA-256 hash is a 64-character code computed from a file's content. If one character in the file changes, the hash changes. The helper first turns Windows line endings into the Mac and Linux form. So it is a **line-ending-normalized text hash**: everyone gets the same hash for the same data. Use it for text files such as CSV files, notebooks and reports. For a binary file, such as an image, hash the bytes unchanged.
- the **random seed**, the number that fixes any random draw (here 1404).

In [ ]:
DATA = Path("../data/course/cfpb_complaints.csv")
SEED = 1404  # the seed for every random draw in this notebook

def text_sha256(path):
    """Take the path of a text file; return its SHA-256 hash (64 characters), with Windows line endings changed to the Mac and Linux form first."""
    # Windows line endings (\r\n) become \n, so the hash is the same on Windows and Mac.
    data = Path(path).read_bytes().replace(b"\r\n", b"\n")
    # hexdigest() writes the hash as text.
    return hashlib.sha256(data).hexdigest()

data_hash = text_sha256(DATA)
print("rows:", len(pd.read_csv(DATA)))
print("sha256:", data_hash[:12])  # the first 12 characters are enough to compare by eye

### 1.4 The analysis as one function that starts from the raw file

A result is easier to rerun when one function goes from the raw file to the table. This one keeps the credit card complaints, flags those that mention a fee (the word *fee* or *fees*), and counts monetary relief in each group. `relief` is the count with the outcome; `complaints` is the denominator.

In [ ]:
# \b marks the edge of a word, so "fee" does not match inside "feedback"; (?:fee|fees) means fee or fees.
FEE_PATTERN = r"\b(?:fee|fees)\b"

def fee_analysis(path, product="Credit card"):
    """Take the data file and a product; return relief counts and complaint counts for each fee group."""
    data = pd.read_csv(path)
    data = data[data["product"].eq(product)].copy()
    # Two True/False columns: does the narrative mention a fee, and was the complaint closed with money back?
    data["mentions_fee"] = data["narrative"].str.contains(FEE_PATTERN, case=False, regex=True)
    data["monetary_relief"] = data["company_response"].eq("Closed with monetary relief")
    # For each fee group: relief = the number of True values (sum), complaints = the number of rows (count).
    table = data.groupby("mentions_fee")["monetary_relief"].agg(relief="sum", complaints="count")
    # Readable row names instead of True and False.
    table = table.rename(index={True: "mentions a fee", False: "no fee mention"})
    return table.sort_index()

first = fee_analysis(DATA)
first

### 1.5 Read before you count

A keyword flag is a **proxy**: it stands in for "this complaint is about a fee", but it can be wrong. *No fee was charged* also matches. Read a few flagged complaints before you trust the count. The cell shows the words around the first match in three complaints drawn with the seed.

In [ ]:
def around(text, width=70):
    """Take a text; return the words around its first fee match (width characters on each side)."""
    match = re.search(FEE_PATTERN, text, flags=re.IGNORECASE)
    # match.start() and match.end() are the positions of the match; max(0, ...) avoids a negative start.
    return "..." + text[max(0, match.start() - width): match.end() + width] + "..."

# query keeps the rows that meet the condition.
card = pd.read_csv(DATA).query("product == 'Credit card'")
flagged = card[card["narrative"].str.contains(FEE_PATTERN, case=False, regex=True)]
# random_state=SEED draws the same 3 complaints on every run.
examples = flagged.sample(3, random_state=SEED)
for cid, text in zip(examples["complaint_id"], examples["narrative"]):
    print(cid, around(text))

### 1.6 Rerun from a fresh state

A notebook can give a result that you cannot get again: a cell was run twice, run out of order, or used a variable that you later deleted. The test is to rerun from a **fresh state**: in VS Code choose **Restart**, then **Run All**. Do this for your project before you submit.

Inside one run, you can also check that the function gives the same table twice. A short fingerprint of the result table makes the comparison easy to record.

In [ ]:
second = fee_analysis(DATA)
# The table as CSV text (with \n line endings), turned into bytes and hashed.
result_csv = first.to_csv(lineterminator="\n")
result_hash = hashlib.sha256(result_csv.encode()).hexdigest()[:12]
# equals is True when the two tables have the same labels and values.
print("same table on rerun:", first.equals(second))
print("result fingerprint:", result_hash)

### 1.7 Random draws need a seed

Suppose you read a random sample of 30 fee complaints. Without a seed, each run draws a different sample, so a number from it changes every time. With `random_state=SEED`, the draw is the same on every run and every computer.

In [ ]:
fee_cases = card[card["narrative"].str.contains(FEE_PATTERN, case=False, regex=True)].copy()
fee_cases["monetary_relief"] = fee_cases["company_response"].eq("Closed with monetary relief")
# Two draws of 30 without a seed, then two with the seed; each time, count the complaints with relief.
no_seed = []
for _ in range(2):
    no_seed.append(int(fee_cases.sample(30)["monetary_relief"].sum()))
with_seed = []
for _ in range(2):
    with_seed.append(int(fee_cases.sample(30, random_state=SEED)["monetary_relief"].sum()))
print("relief in a sample of 30, no seed:  ", no_seed, "(can change from run to run)")
print("relief in a sample of 30, seed 1404:", with_seed)

### 1.8 A readiness checklist

The helper `readiness_checks` turns the record into a table. It checks that each file exists and gives its size and its line-ending-normalized text hash (meant for text files). It also checks that a seed is recorded and the AI-use statement is written, and it lists the package versions. A row marked `CHECK` needs your attention. You will run the same helper on your own project in Part 2.

In [ ]:
def readiness_checks(files, seed, ai_use):
    """Take a list of files, a seed and an AI-use statement; return a checklist table (status ok or CHECK)."""
    rows = []
    for f in files:
        f = Path(f)
        if f.exists():
            # Size counted with \n line endings, as in text_sha256; {size:,} adds thousands commas.
            size = len(f.read_bytes().replace(b"\r\n", b"\n"))
            rows.append({"check": "file: " + f.name, "status": "ok",
                         "detail": f"{size:,} bytes, sha256 {text_sha256(f)[:12]}"})
        else:
            rows.append({"check": "file: " + f.name, "status": "CHECK", "detail": "not found: " + str(f)})

    if seed is not None:
        seed_status = "ok"
    else:
        seed_status = "CHECK"
    rows.append({"check": "random seed recorded", "status": seed_status, "detail": str(seed)})

    # strip() removes spaces at both ends, so a statement of only spaces counts as empty.
    statement = ai_use.strip()
    if statement:
        rows.append({"check": "AI-use statement written", "status": "ok", "detail": statement[:60]})
    else:
        rows.append({"check": "AI-use statement written", "status": "CHECK", "detail": "empty"})

    # The versions as one text, such as "python 3.13.1, numpy 2.3.0, ...".
    version_parts = []
    for name, version in versions.items():
        version_parts.append(f"{name} {version}")
    rows.append({"check": "package versions recorded", "status": "ok", "detail": ", ".join(version_parts)})
    return pd.DataFrame(rows)

example_files = [DATA, Path("../data/course/README.md"), Path("M14_Project_Readiness.ipynb")]
example_checks = readiness_checks(example_files, SEED, "none")
example_checks

### 1.9 Show the denominator

"33% of fee complaints got money back" invites the question: 33% of how many? A share without its denominator can come from 3 complaints or from 3,000. Write the count, the denominator and the share together.

In [ ]:
def share_label(k, n):
    """Take a count and a denominator; return a text such as "97 of 291 (33%)"."""
    # :, adds thousands commas; :.0% writes a share as a percent without decimals.
    return f"{k:,} of {n:,} ({k / n:.0%})"

# iterrows gives the rows one at a time, as (row label, row) pairs.
for group, row in first.iterrows():
    percent_only = row["relief"] / row["complaints"]
    # {group:>15} right-aligns the group name in 15 characters.
    print(f"{group:>15}: percent only {percent_only:.0%}  |  with denominator {share_label(row['relief'], row['complaints'])}")

### 1.10 Add an uncertainty interval

A share from a sample is an estimate. A 95% **confidence interval** (CI) gives the range of shares that fit the data. For a proportion we use the **Wilson interval**. It behaves well for small groups and for shares near 0% or 100%, where the simple "p ± 1.96 × standard error" interval can go below 0 or above 100%.

`add_intervals` adds the share, the interval and a readable label. The last line checks one interval against `scipy`.

In [ ]:
def wilson(k, n, z=1.96):
    """Take a count k and a denominator n; return the low and high ends of the Wilson interval (95% with z = 1.96)."""
    p = k / n
    # The centre of the interval is pulled a little towards 50%, more so for small n.
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    # Half the width of the interval.
    half = z / (1 + z**2 / n) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return centre - half, centre + half

def add_intervals(table, k, n):
    """Take a table and the names of its count and denominator columns; return it with share, low, high and label."""
    out = table.copy()
    out["share"] = out[k] / out[n]
    # wilson works on whole columns at once and returns two columns: the low ends and the high ends.
    out["low"], out["high"] = wilson(out[k], out[n])
    labels = []
    for count, total, low, high in zip(out[k], out[n], out["low"], out["high"]):
        labels.append(f"{share_label(count, total)}, 95% CI {low:.0%} to {high:.0%}")
    out["label"] = labels
    return out

exhibit = add_intervals(first, k="relief", n="complaints")
print(exhibit["label"].to_string())
# Check one interval against the Wilson interval in scipy.
from scipy.stats import binomtest
k, n = first.loc["mentions a fee", ["relief", "complaints"]]
print("scipy check:", binomtest(int(k), int(n)).proportion_ci(method="wilson"))

### 1.11 A small sample overclaims

The seeded sample of 30 from step 1.7 gives a share too. Compare its interval with the interval from all fee complaints. The small sample's interval is much wider. A report that gives only the sample's share would sound more certain than the evidence is.

In [ ]:
k30 = with_seed[0]  # the relief count in the seeded sample of 30
lo30, hi30 = wilson(k30, 30)
fee_row = exhibit.loc["mentions a fee"]
# Interval widths in percentage points.
width30 = (hi30 - lo30) * 100
width_all = (fee_row["high"] - fee_row["low"]) * 100
print(f"sample of 30:       {share_label(k30, 30)}, 95% CI {lo30:.0%} to {hi30:.0%}, width {width30:.0f} points")
print(f"all fee complaints: {fee_row['label']}, width {width_all:.0f} points")

### 1.12 An interval for any metric: the bootstrap

Wilson works for one share, k of n. Many project results are not a single share: macro F1, precision at 5, or the difference between two methods. For these, use the **bootstrap** from Lab 13 in a general form.

`bootstrap_interval` takes a table with one row per record and a function that computes your metric from such a table. It draws the records at random *with replacement* 2,000 times, recomputes the metric on each draw, and returns the value with the middle 95% of the draws.

First a check on a share we know: for the 291 fee complaints, the bootstrap and the Wilson interval should almost agree.

In [ ]:
def bootstrap_interval(records, metric, reps=2000, seed=SEED):
    """Take a table (one row per record) and a metric function; return the metric with its 95% bootstrap interval."""
    # A random number generator with a fixed seed, so everyone gets the same draws.
    rng = np.random.default_rng(seed)
    records = records.reset_index(drop=True)
    draws = []
    for _ in range(reps):
        # len(records) random row positions, drawn with replacement; iloc picks those rows.
        positions = rng.integers(0, len(records), len(records))
        draws.append(metric(records.iloc[positions]))
    # The middle 95% of the draws.
    low, high = np.percentile(draws, [2.5, 97.5])
    return {"value": metric(records), "low": low, "high": high}

def relief_share(records):
    """Take a table of complaints; return the share closed with monetary relief."""
    # The mean of a True/False column is the share of True values.
    return records["monetary_relief"].mean()

boot = bootstrap_interval(fee_cases, relief_share)
wilson_low, wilson_high = wilson(int(fee_cases["monetary_relief"].sum()), len(fee_cases))
print(f"bootstrap: {boot['value']:.1%}, 95% interval {boot['low']:.1%} to {boot['high']:.1%}")
print(f"Wilson:    {boot['value']:.1%}, 95% interval {wilson_low:.1%} to {wilson_high:.1%}")

Now a metric that is not a share. In Lab 8, two classifiers routed complaints to six products, and Lab 9 opened their test predictions once. The file `data/predictions/m08_route_predictions.csv` holds them. Macro F1 is computed from all records at once, so it has no Wilson interval, but the bootstrap works.

For the **difference** between two methods, compute both on the same draw. Each complaint then keeps its two predictions together, as in the paired comparison of Lab 13. If the interval of the difference includes 0, the data cannot tell the two methods apart. This cell takes about half a minute.

In [ ]:
from sklearn.metrics import f1_score

# The test predictions of the two Lab 8 classifiers, one row per complaint.
PREDICTIONS = Path("../data/predictions/m08_route_predictions.csv")
routes = pd.read_csv(PREDICTIONS).query("split == 'test'")

def macro_f1(records, column="tfidf_route"):
    """Take a table of predictions and a prediction column; return macro F1 against the true route."""
    # average="macro": the F1 of each product, then the plain average of the six.
    return f1_score(records["route"], records[column], average="macro")

def f1_gap(records):
    """Take a table of predictions; return TF-IDF macro F1 minus embedding macro F1 on the same records."""
    return macro_f1(records, "tfidf_route") - macro_f1(records, "emb_route")

# Each call returns a dictionary with the value and the low and high ends of its interval.
f1_result = bootstrap_interval(routes, macro_f1)
gap_result = bootstrap_interval(routes, f1_gap)
# One column per metric; .T flips the table, so each metric is a row.
metric_table = pd.DataFrame({"TF-IDF macro F1": f1_result, "TF-IDF minus embedding": gap_result}).T
metric_table["n"] = len(routes)
metric_table.round(3)

In your project, pass your own table of test predictions (one row per record) and your own metric function. Examples are precision at 5 averaged over queries, or the share of correct fields. Report the value, the interval and n together.

### 1.13 Build the exhibit

A business exhibit should be readable on its own:

- the **title states the finding**, not just the variables;
- each group shows its **count and denominator**;
- the **interval** is drawn, so the reader sees the uncertainty;
- a **note** gives the data source and the main limitation.

`plot_exhibit` draws a dot for each share and a line for its interval, then saves the figure. For a metric that is not a share, such as macro F1, set `percent=False`. The axis then fits the intervals, and a difference that can be negative gets a dashed line at 0.

A chart also needs **alt text**: one or two sentences that give the finding and the key numbers for a reader who cannot see it. The cell prints an example. Put the same text beside the chart in your report and slides.

In [ ]:
def plot_exhibit(table, title, xlabel, note, path, percent=True):
    """Take a table with share, low, high and label; draw a dot and an interval line per row and save the figure."""
    # The figure gets taller with more rows.
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(table) + 2.4))
    # One y position per row; [::-1] reverses them, so the first row is at the top.
    y = np.arange(len(table))[::-1]
    # A line for each interval and a dot for each share.
    ax.hlines(y, table["low"], table["high"], color="#E87722", linewidth=4)
    ax.plot(table["share"], y, "o", color="#1C2E45", markersize=9)
    if percent:
        # Shares run from 0% to 100%.
        x_low, x_high = 0, 1
    else:
        # Other metrics, such as a difference, can be negative: fit the axis to the intervals.
        x_low = table["low"].min()
        x_high = table["high"].max()
        if x_low < 0:
            # Keep 0 in view and draw a reference line there: an interval that crosses it cannot separate the two sides.
            x_high = max(x_high, 0)
            ax.axvline(0, color="grey", linewidth=1, linestyle="--")
        width = x_high - x_low
        # Some room on the left, and more on the right for the labels.
        x_low, x_high = x_low - 0.1 * width, x_high + 0.6 * width
    # The label just right of each interval.
    gap = 0.01 * (x_high - x_low)
    for y_pos, high, label in zip(y, table["high"], table["label"]):
        ax.text(high + gap, y_pos, label, va="center", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(table.index)
    ax.set_ylim(-0.6, len(table) - 0.4)
    ax.set_xlim(x_low, x_high)
    if percent:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    # textwrap.fill breaks long text into lines of at most 80 (or 120) characters.
    ax.set_xlabel(textwrap.fill(xlabel, 80))
    fig.suptitle(textwrap.fill(title, 80), x=0.01, ha="left", fontsize=12)
    fig.text(0.01, 0.01, textwrap.fill(note, 120), fontsize=8, va="bottom")
    fig.tight_layout(rect=(0, 0.1, 1, 0.92))
    fig.savefig(path, dpi=150)
    plt.show()

plot_exhibit(exhibit,
             title="Credit card complaints that mention a fee were closed with money back about twice as often",
             xlabel="Share closed with monetary relief (dot) and 95% Wilson interval (line)",
             note="Source: 1,500 credit card complaints in the BAN 6403 course data (CFPB, July-December 2025). Fee = the word fee or fees. Complainants are not a random sample of customers.",
             path=OUTPUT_DIR / "fee_relief_exhibit.png")

# Alt text: the finding and the key numbers in words, for readers who cannot see the chart.
alt_text = ("Dot and interval chart of credit card complaints closed with monetary relief. "
            + "Complaints that mention a fee: " + exhibit.loc["mentions a fee", "label"] + ". "
            + "No fee mention: " + exhibit.loc["no fee mention", "label"] + ".")
print("Alt text:", alt_text)

### 1.14 Evidence, interpretation, recommendation

Write the result in three layers. Each layer must be supported by the one before it.

- **Evidence** (what the data show, with denominators and intervals) comes from the cell below.
- **Interpretation** (what it means, and what it does not): fee complaints and monetary relief go together in this data. That is an association. It does not show that fees cause refunds, and the flag is a keyword proxy. Only people who chose to complain are in the data.
- **Recommendation** (who should do what, and how to check it): the card operations lead could pilot a priority review queue for fee complaints. Then the lead could check whether review time and monetary relief change, and read a sample of flagged complaints to measure how often the flag is right.

In [ ]:
# The evidence layer: both labels and the ratio of the two shares.
nofee_row = exhibit.loc["no fee mention"]
print("Credit card complaints closed with monetary relief")
print("  mention a fee: ", fee_row["label"])
print("  no fee mention:", nofee_row["label"])
print("Ratio of the two shares:", round(fee_row["share"] / nofee_row["share"], 2))

### 1.15 Save your outputs

This writes the exhibit table, the interval table from step 1.12, the readiness checklist and a small manifest to `outputs/m14`. The manifest records both input files with their hashes (the complaints and the Lab 8 predictions), the seed, the versions and the result fingerprint. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
exhibit.to_csv(OUTPUT_DIR / "fee_relief_exhibit.csv")
metric_table.to_csv(OUTPUT_DIR / "bootstrap_example.csv")
example_checks.to_csv(OUTPUT_DIR / "readiness_example.csv", index=False)
# The record a reader needs to rerun the example: both input files with their hashes, the seed and the versions.
manifest = {"data_file": "data/course/cfpb_complaints.csv", "data_sha256": data_hash,
            "predictions_file": "data/predictions/m08_route_predictions.csv", "predictions_sha256": text_sha256(PREDICTIONS),
            "seed": SEED, "versions": versions, "result_fingerprint": result_hash}
(OUTPUT_DIR / "manifest_example.json").write_text(json.dumps(manifest, indent=2))
print("saved in outputs/m14: fee_relief_exhibit.png, fee_relief_exhibit.csv, bootstrap_example.csv,",
      "readiness_example.csv, manifest_example.json")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. In step 1.11, why is the interval for the sample of 30 so much wider than the interval for all fee complaints? What would you write if the sample of 30 were your only evidence?
2. The exhibit shows that fee complaints were closed with monetary relief more often. Give one reason why this does not prove that fees cause refunds.

**Your answer:**

## Part 2 Practice: your own project

Now apply the same checks to your own project. If you do not have project files on this computer yet, leave the settings below as they are: every cell then uses the course data, so Run All still works. Use your project as soon as you can; the checks are meant for it.

### 2.1 Your project settings

Edit the three lines below:

- `PROJECT_FILES`: the files a reader needs to rerun your project, such as your data file(s), your main notebook and your report. Use paths relative to this repository (for example `"project/data/private/reviews.csv"` or `"project/notebooks/02_baseline.ipynb"`) or full paths (for example `"C:/Users/you/Documents/GitHub/ban6403-jdoe/project/report/report.html"`).
- `PROJECT_SEED`: the random seed your project uses, or leave `None` if your project has no random step yet.
- `AI_USE_STATEMENT`: one or two sentences on how you used AI tools in the project and how you checked the results ("none" is fine).

In [ ]:
# Edit these three lines for your project. Leave them unchanged to use the course example.
PROJECT_FILES = []
PROJECT_SEED = None
AI_USE_STATEMENT = ""

### 2.2 Run the readiness checklist

The notebook runs from the `notebooks` folder, so this cell puts `../` in front of each path (a full path stays as it is). Then it runs `readiness_checks` from step 1.8. With no project files listed, it checks the course data instead. Expect some rows marked `CHECK` the first time.

In [ ]:
# The repository is one folder up (".."); joining ".." with a full path keeps the full path.
project_files = []
for f in PROJECT_FILES:
    project_files.append(Path("..") / f)
using_course_example = len(project_files) == 0
if using_course_example:
    project_files = [DATA, Path("../data/course/README.md")]
print("using the course example:", using_course_example)
project_checks = readiness_checks(project_files, PROJECT_SEED, AI_USE_STATEMENT)
n_check = (project_checks["status"] == "CHECK").sum()
print("rows to check:", n_check, "of", len(project_checks))
project_checks

### Exercise 1: fix one reproducibility issue and keep a revision log

1. Pick at least one row marked `CHECK` in 2.2, or another reproducibility problem you know about in your project (for example a cell that only works after another cell is run twice, or a hard-coded path that only exists on your computer). Fix it in your project or in the settings in 2.1, then rerun 2.1 and 2.2.
2. Build a table `revision_log` with the columns `item`, `problem`, `fix` and `source`, and at least two rows: one from the readiness check, and one from feedback on your project. Your MS3 feedback may not be back yet. Then use this week's peer review, or a finding from the project clinic self-check (Flex Meeting 1 or the Project Clinic page). If you use the course example, log the seed and the AI-use statement.
3. Save it as `outputs/m14/revision_log.csv` and display it.

In [ ]:
# Exercise 1: your code here
# TODO: fix at least one CHECK row (or another reproducibility problem), then rerun 2.1 and 2.2.
# TODO: build revision_log with the columns item, problem, fix and source (at least two rows).
# TODO: save it as OUTPUT_DIR / "revision_log.csv" and display it.

### Exercise 2: rerun your headline number from a fresh state

1. Write a function `headline()` that reads a file from disk and returns the main number of your report (a count, a share, an F1 score ...). Use your seed for any random step. Read your raw data file, or, if your headline is a model score, the saved test predictions with record IDs (for example `project/outputs/ms3/test_predictions.csv`), and recompute the metric from them. You do not need to refit your model here. If you use the course example: among **checking or savings account** complaints that mention a fee, return the share closed with monetary relief. The last lines of `headline()` can be:

   ```python
   table = fee_analysis(DATA, product="Checking or savings account")
   row = table.loc["mentions a fee"]
   return row["relief"] / row["complaints"]
   ```
2. Call `headline()` twice and print whether the two results are equal.
3. Set `REPORTED_VALUE` to the number in your current report draft and print both. If they differ, write in a comment which one is right and why. With the course example and no report draft, write the share as a report would show it, rounded to two decimals, and compare it with the rerun.
4. Restart the kernel and run the whole notebook once more before you submit.

In [ ]:
# Exercise 2: your code here
# TODO: write headline() that reads a file from disk and returns your report's main number.
# TODO: call headline() twice and print whether the two results are equal.
# TODO: set REPORTED_VALUE to the number in your report draft (course example: the share rounded to two decimals)
#       and print both.

### Exercise 3: finalize one exhibit with denominators and uncertainty

1. Build a table with one row per group and two count columns for your key result: the count with the outcome and the group size (the denominator). The row labels should be readable group names.
2. Add intervals with `add_intervals(table, k=..., n=...)`.
3. Draw it with `plot_exhibit`. Write a title that states the finding and a note that gives the data source, the denominators and the main limitation. Save it as `outputs/m14/my_exhibit.png`, and save the table as `outputs/m14/my_exhibit.csv`.
4. Write alt text for the chart in `MY_ALT_TEXT`: the finding and the key numbers in one or two sentences, as in step 1.13. Print it, and use the same text beside the chart in your report and slides.

If you use the course example, show for each of the six products the share of fee-mentioning complaints that were closed with monetary relief. Hint: build `fee_analysis(DATA, product=p).loc["mentions a fee"]` for each product `p`. If your key result is not a share (for example macro F1 or precision at 5), use `bootstrap_interval` from step 1.12 for each method or group. Build a table with the columns `share` (the metric value), `low`, `high` and `label`, such as "macro F1 0.82, 95% CI 0.81 to 0.84, n = 2,000". Draw it with `plot_exhibit(..., percent=False)`.

In [ ]:
# Exercise 3: your code here
# TODO: build a table with one row per group: the count with the outcome and the group size.
# TODO: add intervals with add_intervals(table, k=..., n=...).
# TODO: draw it with plot_exhibit (title = the finding, note = source, denominators, limitation);
#       save OUTPUT_DIR / "my_exhibit.png" and the table as OUTPUT_DIR / "my_exhibit.csv".
# TODO: write MY_ALT_TEXT (the finding and the key numbers in one or two sentences) and print it.

### Practice interpretation

Write 3–5 sentences for your project (or the course example):

- State your recommendation in three layers: the evidence (with a denominator and an interval from Exercise 3), the interpretation, and the action for a named decision owner.
- Name the main limitation a reader must know.
- Say what you fixed in Exercise 1 and whether your headline number reproduced in Exercise 2.

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, or to draft text, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. An interval for the difference in shares

The exhibit gives an interval for each share. A decision often needs the difference between the two shares. Use `bootstrap_interval` from step 1.12 on the 1,500 credit card complaints (`card`). First add the columns `mentions_fee` and `monetary_relief`, as `fee_analysis` does. Then write a metric function that returns the relief share of fee complaints minus the relief share of the others. Does the interval include zero?

In [ ]:
# Challenge A: your code here
# TODO: add the columns mentions_fee and monetary_relief to card, as fee_analysis does.
# TODO: write a metric function: relief share of fee complaints minus relief share of the others.
# TODO: call bootstrap_interval(card_with_flags, your_function) and check whether the interval includes 0.

### B. Rerun from a fresh clone

Reproducibility means another person can rerun your work. Clone your repository into a new folder (GitHub Desktop: **File > Clone repository**), create the environment from your environment file, and run your main notebook from the top. Compare the hashes of your main text outputs with `text_sha256`. For binary outputs, such as `.png` images, hash the bytes unchanged: `hashlib.sha256(Path(p).read_bytes()).hexdigest()`. List anything you had to fix.

### C. One additional modeling technique

If your project has time left, try one additional method from the course (for example an embedding classifier next to your TF-IDF baseline). Choose the method and its settings on development data only (train and validation). Then compare it with the original once on the frozen test set, with the same metric and denominators. A comparison you design or tune after seeing test results is exploratory: label it so in your report. Keep your recommendation tied to the evidence you trust most.

## Further reading

- [Sandve and others (2013), Ten Simple Rules for Reproducible Computational Research, PLOS Computational Biology](https://doi.org/10.1371/journal.pcbi.1003285)
- [The Turing Way: Guide for reproducible research](https://book.the-turing-way.org/reproducible-research/reproducible-research)
- [Brown, Cai and DasGupta (2001), Interval Estimation for a Binomial Proportion, Statistical Science](https://doi.org/10.1214/ss/1009213286)

Next, Module 15: you record your presentation, answer questions on it, and submit the final project package.